# SoloMate AI — LangGraph 설계 문서

## 1. 에이전트 이름

**SoloMate AI** (솔로 연습 보조 그래프)

## 2. 목적

기타·악기 연습자가 **코드 진행과 분위기**를 입력하면, 시스템이 이를 정리·요약하고 **스케일·프레이즈·애드립 방향** 같은 연습용 힌트를 한 번에 제안하는 문제를 해결합니다. LLM 없이도 동작하는 **결정적(deterministic) 파이프라인**으로 LangGraph 노드·엣지 구조를 익히기 위한 최소 뼈대입니다.

## 3. 핵심 기능 (3가지 이상)

1. **코드 진행·분위기 파싱**: 마지막 사용자 메시지에서 첫 줄을 코드 진행, 이후 줄을 분위기/메모로 해석합니다.
2. **분석 요약**: `analyze_progression` 노드가 `chord_progression`, `mood`, `analysis_summary`를 상태에 채웁니다.
3. **연습 제안**: `recommend_practice` 노드가 스케일·프레이즈 힌트를 묶어 `practice_output`과 `AIMessage`로 반환합니다.
4. **상태 스키마**: `SoloMateState`가 `MessagesState`를 확장해 도메인 필드를 보관합니다.

## 4. 그래프 구조 (노드·엣지)

```
  START
    |
    v
 analyze_progression
    |
    v
 recommend_practice
    |
    v
   END
```

- **노드**: `analyze_progression` → `recommend_practice`
- **엣지**: 선형 파이프라인 (`START` → … → `END`)


## 5. 구현 코드

아래 셀은 프로젝트 루트의 `main.py`와 동일한 로직입니다.

In [ ]:
"""
SoloMate AI — LangGraph 기초 그래프.
코드 진행 입력 → 분석 → 연습용 제안(스케일·프레이즈 힌트) 흐름의 최소 뼈대.
"""

from __future__ import annotations

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from typing_extensions import NotRequired


class SoloMateState(MessagesState):
    """대화(messages)와 기타 연습용 도메인 상태."""

    chord_progression: NotRequired[str]
    mood: NotRequired[str]
    analysis_summary: NotRequired[str]
    practice_output: NotRequired[str]


def _last_user_text(state: SoloMateState) -> str:
    msgs = state.get("messages") or []
    for m in reversed(msgs):
        if isinstance(m, HumanMessage):
            c = m.content
            return c if isinstance(c, str) else str(c)
    return ""


def analyze_progression(state: SoloMateState) -> dict:
    """사용자 메시지에서 코드 진행·분위기를 추출하고 분석 요약을 만든다."""
    text = _last_user_text(state).strip()
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    chords = lines[0] if lines else "(입력 없음)"
    mood = " / ".join(lines[1:]) if len(lines) > 1 else "미지정"

    summary = (
        f"입력된 코드 진행(추정): {chords}\n"
        f"분위기/메모: {mood}\n"
        "다음 노드에서 스케일·애드립·연습 프레이즈 방향을 제안합니다."
    )
    return {
        "chord_progression": chords,
        "mood": mood,
        "analysis_summary": summary,
    }


def recommend_practice(state: SoloMateState) -> dict:
    """분석 결과를 바탕으로 연습용 힌트와 AI 메시지를 생성한다."""
    chords = state.get("chord_progression") or ""
    mood = state.get("mood") or ""

    scale_hint = (
        "다이아토닉/펜타토닉을 1차 후보로 두고, "
        "IV·V 구간에서는 코드톤(루트·3도·5도)과 근접 반음 앱로치를 섞어 보세요."
    )
    phrase_hint = (
        "프레이즈: 상행 3~4음 스케일 런 후 타겟 코드의 루트 또는 3도로 착지(느린 템포로 반복).\n"
        "팁: 코드 체인지 직전 한 박은 이전 화성의 패싱톤으로 정리하면 자연스럽습니다."
    )
    practice = (
        f"[SoloMate 연습 제안]\n"
        f"코드 진행: {chords}\n"
        f"분위기: {mood}\n\n"
        f"스케일·애드립: {scale_hint}\n\n"
        f"{phrase_hint}"
    )
    return {
        "practice_output": practice,
        "messages": [AIMessage(content=practice)],
    }


def build_solomate_graph():
    graph = StateGraph[SoloMateState, None, SoloMateState, SoloMateState](SoloMateState)
    graph.add_node("analyze_progression", analyze_progression)
    graph.add_node("recommend_practice", recommend_practice)
    graph.add_edge(START, "analyze_progression")
    graph.add_edge("analyze_progression", "recommend_practice")
    graph.add_edge("recommend_practice", END)
    return graph.compile()


graph = build_solomate_graph()

## 6. 실행 예시

첫 줄은 코드 진행, 이후 줄은 분위기·메모로 넣습니다.

In [ ]:
demo = graph.invoke(
    {
        "messages": [
            HumanMessage(
                "C | Am | F | G\n"
                "발라드, 첫 솔로라 부담 없이 가고 싶어."
            )
        ],
    }
)
print(demo.get("analysis_summary", ""))
print("---")
print(demo.get("practice_output", ""))

## 7. 그래프 시각화 (선택)

`draw_ascii()`는 LangGraph 버전에 따라 제공됩니다.

In [ ]:
g = graph.get_graph()
if hasattr(g, "draw_ascii"):
    print(g.draw_ascii())
else:
    print("이 환경에서는 draw_ascii를 사용할 수 없습니다.")